# EDA — Risk Profiling

Carga de los 3 datasets y vista rápida de sus datos.

In [32]:
import pandas as pd

DATA = "../data/"

users = pd.read_csv(DATA + "user_inventory.csv", parse_dates=["created_at"])
perms = pd.read_csv(DATA + "permission_inventory.csv", parse_dates=["assigned_at", "expires_at"])
logs  = pd.read_csv(DATA + "access_logs.csv", parse_dates=["timestamp"])

TODAY = pd.Timestamp("2026-06-01")  # fecha de referencia del análisis

In [33]:
# Descripciones de campos (según docs/CHALLENGE.md), adjuntas a cada df via .attrs
users.attrs["descriptions"] = {
    "user_id":    "Identificador único",
    "user_type":  "Internal o External",
    "department": "Departamento al que pertenece",
    "role":       "Rol dentro de la empresa",
    "status":     "Active o Inactive",
    "created_at": "Fecha de creación de la cuenta",
    "manager_id": "user_id de su manager (puede ser null)",
}
perms.attrs["descriptions"] = {
    "user_id":       "Usuario al que se le asignó el permiso",
    "resource_id":   "Identificador del recurso",
    "resource_type": "admin_panel, payment_portal, vdi, database, api_internal, drive, email_system, reporting_tool",
    "criticality":   "VERY_HIGH, HIGH, MEDIUM, LOW",
    "assigned_at":   "Fecha de asignación",
    "expires_at":    "Fecha de expiración (null = sin vencimiento)",
}
logs.attrs["descriptions"] = {
    "user_id":              "Usuario que realizó el acceso",
    "timestamp":            "Fecha y hora del acceso",
    "resource_id":          "Recurso al que accedió",
    "resource_type":        "Tipo de recurso",
    "action":               "READ, WRITE, DELETE, EXPORT, LOGIN, QUERY",
    "source_system":        "Sistema de origen del evento",
    "session_duration_sec": "Duración de la sesión en segundos",
}

def describe_cols(df):
    """Tabla con dtype, nulos y descripción de cada columna."""
    return pd.DataFrame({
        "dtype":       df.dtypes.astype(str),
        "nulos":       df.isna().sum(),
        "descripcion": pd.Series(df.attrs.get("descriptions", {})),
    })

## user_inventory

In [34]:
print(users.shape)
display(users.head())
describe_cols(users)

(500, 7)


,user_id,user_type,department,role,status,created_at,manager_id
0,USR0001,External,Operations,Reps,Active,2025-02-09,NaN
1,USR0002,Internal,Fintech,Specialist,Active,2024-12-02,NaN
2,USR0003,External,Engineering,Reps,Active,2025-01-19,NaN
3,USR0004,External,Operations,Reps,Active,2025-01-12,NaN
4,USR0005,Internal,Operations,Engineer,Active,2025-04-13,NaN


,dtype,nulos,descripcion
user_id,str,0,Identificador único
user_type,str,0,Internal o External
department,str,0,Departamento al que pertenece
role,str,0,Rol dentro de la empresa
status,str,0,Active o Inactive
created_at,datetime64[us],0,Fecha de creación de la cuenta
manager_id,str,50,user_id de su manager (puede ser null)


## permission_inventory

In [35]:
print(perms.shape)
display(perms.head())
describe_cols(perms)

(2944, 6)


,user_id,resource_id,resource_type,criticality,assigned_at,expires_at
0,USR0001,DRI005,drive,MEDIUM,2025-04-21,NaT
1,USR0001,EMA002,email_system,MEDIUM,2025-05-08,NaT
2,USR0001,EMA010,email_system,MEDIUM,2025-03-04,NaT
3,USR0001,DRI007,drive,MEDIUM,2025-08-19,2026-08-07
4,USR0002,DRI006,drive,MEDIUM,2025-10-23,NaT


,dtype,nulos,descripcion
user_id,str,0,Usuario al que se le asignó el permiso
resource_id,str,0,Identificador del recurso
resource_type,str,0,"admin_panel, payment_portal, vdi, database, ap..."
criticality,str,0,"VERY_HIGH, HIGH, MEDIUM, LOW"
assigned_at,datetime64[us],0,Fecha de asignación
expires_at,datetime64[us],2004,Fecha de expiración (null = sin vencimiento)


## access_logs

In [36]:
print(logs.shape)
display(logs.head())
describe_cols(logs)

(20495, 7)


,user_id,timestamp,resource_id,resource_type,action,source_system,session_duration_sec
0,USR0454,2025-12-20 12:50:02,API007,api_internal,DELETE,api_internal,1165
1,USR0253,2026-03-02 09:46:11,DRI005,drive,QUERY,drive,775
2,USR0456,2026-04-14 17:58:45,EMA006,email_system,LOGIN,email_system,1458
3,USR0247,2026-03-16 17:01:51,API010,api_internal,WRITE,api_internal,1798
4,USR0125,2026-05-05 11:24:03,REP002,reporting_tool,WRITE,reporting_tool,2017


,dtype,nulos,descripcion
user_id,str,0,Usuario que realizó el acceso
timestamp,datetime64[us],0,Fecha y hora del acceso
resource_id,str,0,Recurso al que accedió
resource_type,str,0,Tipo de recurso
action,str,0,"READ, WRITE, DELETE, EXPORT, LOGIN, QUERY"
source_system,str,0,Sistema de origen del evento
session_duration_sec,int64,0,Duración de la sesión en segundos


# Calidad de datos y relaciones entre tablas

`user_id` enlaza las 3 tablas. `permission_inventory` dice qué *debería* poder hacer cada usuario y `access_logs` lo que *realmente* hizo: comparar ambas es la base para detectar comportamiento anómalo.

In [37]:
# Integridad referencial, duplicados y cobertura entre tablas
uset       = set(users.user_id)
perm_pairs = set(zip(perms.user_id, perms.resource_id))
log_pairs  = set(zip(logs.user_id, logs.resource_id))

quality = {
    "user_id huérfanos en perms":      (~perms.user_id.isin(uset)).sum(),
    "user_id huérfanos en logs":       (~logs.user_id.isin(uset)).sum(),
    "manager_id inexistente":          (~users.manager_id.dropna().isin(uset)).sum(),
    "permisos duplicados (user+rec)":  perms.duplicated(subset=["user_id", "resource_id"]).sum(),
    "logs duplicados":                 logs.duplicated().sum(),
    "usuarios sin permisos":           len(uset - set(perms.user_id)),
    "usuarios sin accesos":            len(uset - set(logs.user_id)),
    "permisos nunca usados":           len(perm_pairs - log_pairs),
    "fechas imposibles (exp<asig)":    int((perms.expires_at < perms.assigned_at).sum()),
}
pd.Series(quality, name="casos").to_frame()

,casos
user_id huérfanos en perms,0
user_id huérfanos en logs,0
manager_id inexistente,0
permisos duplicados (user+rec),0
logs duplicados,0
usuarios sin permisos,0
usuarios sin accesos,0
permisos nunca usados,105
fechas imposibles (exp<asig),0


La **estructura está limpia**: sin huérfanos, sin duplicados, sin fechas imposibles, y todos los usuarios tienen permisos y accesos. Las anomalías no están en la integridad sino en el **comportamiento**. Notas:
- `resource_type` == `source_system` en el 100% de los logs → columna redundante.
- 105 permisos nunca se usan (posible *over-provisioning*) y 2004 permisos no tienen vencimiento.

## Señales de comportamiento anómalo

In [38]:
# 1) Accesos sin permiso asignado  +  2) permisos expirados usados  +  3) actividad de inactivos
logs["has_perm"] = [(u, r) in perm_pairs for u, r in zip(logs.user_id, logs.resource_id)]
no_perm = logs[~logs.has_perm]

pe = perms.dropna(subset=["expires_at"])[["user_id", "resource_id", "expires_at"]]
acc_exp = logs.merge(pe, on=["user_id", "resource_id"]).query("timestamp > expires_at")

inact = set(users.loc[users.status == "Inactive", "user_id"])

print(f"Accesos a (usuario,recurso) SIN permiso : {len(no_perm)} ({100*len(no_perm)/len(logs):.1f}%)")
print(f"Accesos usando un permiso ya EXPIRADO   : {len(acc_exp)}")
print(f"Permisos ya expirados (a {TODAY.date()}): {perms.expires_at.notna().sum() and int((perms.expires_at < TODAY).sum())}")
print(f"Usuarios Inactive con actividad         : {sorted(inact)} → {logs.user_id.isin(inact).sum()} accesos")

Accesos a (usuario,recurso) SIN permiso : 161 (0.8%)
Accesos usando un permiso ya EXPIRADO   : 93
Permisos ya expirados (a 2026-06-01): 40
Usuarios Inactive con actividad         : ['USR0010', 'USR0011', 'USR0012'] → 53 accesos


In [39]:
# Accesos sin permiso: concentrados en recursos sensibles y en pocos usuarios
print("Por tipo de recurso:")
display(no_perm.resource_type.value_counts().rename("accesos_sin_permiso").to_frame())
print("Top usuarios:")
no_perm.user_id.value_counts().head(10).rename("accesos_sin_permiso").to_frame()

Por tipo de recurso:


,accesos_sin_permiso
resource_type,
api_internal,84
vdi,29
payment_portal,25
admin_panel,14
database,9


Top usuarios:


,accesos_sin_permiso
user_id,
USR0080,84
USR0060,27
USR0041,25
USR0040,25


## Distribuciones: volumen, horario y duración de sesión

Para detectar outliers de comportamiento.

In [40]:
hour = logs.timestamp.dt.hour
logs["off_hours"] = (hour < 7) | (hour > 20)

# Perfil por usuario: volumen, % fuera de horario y duración típica de sesión
perfil = logs.groupby("user_id").agg(
    accesos       = ("timestamp", "size"),
    pct_off_hours = ("off_hours", "mean"),
    dur_mediana_s = ("session_duration_sec", "median"),
)
perfil["pct_off_hours"] = (perfil.pct_off_hours * 100).round(1)

print("Resumen de la distribución (todos los usuarios):")
display(perfil.describe().round(1))

print("\nTop 10 por volumen de accesos:")
display(perfil.sort_values("accesos", ascending=False).head(10))

print("\nTop 10 por % de accesos fuera de horario (7-20h):")
display(perfil.sort_values("pct_off_hours", ascending=False).head(10))

Resumen de la distribución (todos los usuarios):


,accesos,pct_off_hours,dur_mediana_s
count,500.0,500.0,500.0
mean,41.0,0.7,1826.8
std,30.3,6.7,301.7
min,8.0,0.0,715.0
25%,29.0,0.0,1632.4
50%,41.0,0.0,1848.8
75%,50.0,0.0,2013.4
max,396.0,100.0,2785.0



Top 10 por volumen de accesos:


,accesos,pct_off_hours,dur_mediana_s
user_id,,,
USR0020,396,0.0,1707.0
USR0021,395,0.0,1811.0
USR0070,295,41.0,1815.0
USR0080,114,0.0,1476.5
USR0203,77,0.0,2071.0
USR0443,76,0.0,2069.0
USR0285,76,0.0,1652.0
USR0215,76,0.0,1911.5
USR0393,75,0.0,1813.0



Top 10 por % de accesos fuera de horario (7-20h):


,accesos,pct_off_hours,dur_mediana_s
user_id,,,
USR0030,18,100.0,2113.5
USR0012,16,56.2,1847.5
USR0010,16,50.0,2116.5
USR0011,21,47.6,2167.0
USR0050,45,46.7,1796.0
USR0070,295,41.0,1815.0
USR0051,48,33.3,1899.5
USR0337,10,0.0,2050.5
USR0338,34,0.0,2318.5


## Validación de H6 (acción × criticidad) y H7 (Externos)

Antes de resumir las hipótesis, se contrastan las dos que requieren un cruce extra: si las acciones destructivas se concentran en recursos críticos (H6) y si los `External` se comportan distinto que los `Internal` (H7).

In [41]:
# === H6: ¿las acciones destructivas se concentran en recursos críticos? ===
crit_map = perms.drop_duplicates("resource_id").set_index("resource_id").criticality
logs["criticality"] = logs.resource_id.map(crit_map)

print("H6 — distribución de criticidad por acción (% de cada acción):")
ct = pd.crosstab(logs.action, logs.criticality, normalize="index") * 100
display(ct[["LOW", "MEDIUM", "HIGH", "VERY_HIGH"]].round(1))

destr = logs[logs.action.isin(["DELETE", "EXPORT"]) & logs.criticality.isin(["HIGH", "VERY_HIGH"])]
print(f"\nAccesos DELETE/EXPORT sobre HIGH/VERY_HIGH: {len(destr)} ({100*len(destr)/len(logs):.1f}% del log)")
print("Top usuarios:")
display(destr.user_id.value_counts().head(10).rename("destruct_sobre_critico").to_frame())

# === H7: ¿se comportan distinto los External que los Internal? ===
logs["user_type"] = logs.user_id.map(users.set_index("user_id").user_type)
h7 = logs.groupby("user_type").agg(
    usuarios        = ("user_id", "nunique"),
    accesos         = ("user_id", "size"),
    pct_sin_permiso = ("has_perm",  lambda s: (~s).mean() * 100),
    pct_off_hours   = ("off_hours", lambda s:   s.mean() * 100),
)
print("\nH7 — External vs Internal:")
display(h7.round(2))

H6 — distribución de criticidad por acción (% de cada acción):


criticality,LOW,MEDIUM,HIGH,VERY_HIGH
action,,,,
DELETE,19.9,35.0,29.8,15.2
EXPORT,18.3,37.1,29.9,14.8
LOGIN,17.7,36.7,30.7,14.9
QUERY,19.0,37.6,27.9,15.5
READ,17.0,36.2,29.2,17.6
WRITE,19.0,37.0,29.3,14.7



Accesos DELETE/EXPORT sobre HIGH/VERY_HIGH: 3105 (15.2% del log)
Top usuarios:


,destruct_sobre_critico
user_id,
USR0021,98
USR0080,58
USR0070,46
USR0379,23
USR0285,23
USR0020,23
USR0047,22
USR0029,22
USR0220,22



H7 — External vs Internal:


,usuarios,accesos,pct_sin_permiso,pct_off_hours
user_type,,,,
External,80,4115,0.00,0.00
Internal,420,16380,0.98,1.24


**Resultado: H6 y H7 quedan descartadas como señales propias.**

- **H6 ❌** — Se esperaba que las acciones destructivas (`DELETE`/`EXPORT`) apuntaran a los recursos críticos. El crosstab muestra lo contrario: la criticidad se reparte casi **uniforme** entre todas las acciones (un `DELETE` toca recursos críticos en la misma proporción que un `READ`). Los usuarios con más destructivas-sobre-críticos son los mismos de alto volumen (USR0021, USR0080, USR0070): es **aritmética del volumen (H3)**, no un patrón dirigido.
- **H7 ❌** — Se esperaba que los `External` fueran más riesgosos. Resultaron **más limpios** que los `Internal` (0% sin permiso y 0% fuera de horario, vs 0.98% y 1.24%). El `user_type` por sí solo no es señal.

En ambos casos, lo que parecía una pista era ruido. Descartarlas con evidencia es parte del EDA y evita arrastrar falsas señales a etapas posteriores.

# Hipótesis de anomalías a detectar

El EDA sugiere que las anomalías están **inyectadas en usuarios de IDs redondos** (`USR0010`–`USR0080`). Para cada tipo de anomalía se plantea una hipótesis y se contrasta con los datos. La columna **Estado** indica si quedó **confirmada** (hay señal real) o **descartada** (lo que parecía señal era ruido):

| # | Anomalía esperada | Estado | Evidencia en el EDA |
|---|----------|--------|---------------------|
| H1 | **Acceso sin permiso** | ✅ confirmada | 161 accesos (0.8%) a recursos no asignados, concentrados en `api_internal`, `vdi`, `payment_portal`, `admin_panel`. Top: USR0080 (84), USR0060 (27), USR0041 (25), USR0040 (25) |
| H2 | **Cuentas inactivas activas** | ✅ confirmada | USR0010/11/12 son `Inactive` pero suman 53 accesos |
| H3 | **Volumen anómalo** | ✅ confirmada | mediana 41 accesos/usuario (p99 = 76), pero USR0020 (396) y USR0021 (395) son ~10× la mediana / ~5× el p99; USR0070 (295) es un tercer outlier |
| H4 | **Horario inusual** | ✅ confirmada | USR0030 100% fuera de horario; USR0010/11/12/50/70 entre 41-56% |
| H5 | **Permisos expirados / sin uso** | ✅ confirmada | 93 accesos con permiso vencido; 40 permisos expirados a la fecha; 105 nunca usados; 2004 sin vencimiento |
| H6 | **Acciones destructivas sobre recursos críticos** | ❌ descartada | **No hay sesgo:** la criticidad se reparte casi uniforme entre todas las acciones (un `DELETE` toca recursos críticos en la misma proporción que un `READ`). Los usuarios con más `DELETE`/`EXPORT` sobre HIGH/VERY_HIGH son los mismos de alto volumen (USR0021, USR0080, USR0070): es un **efecto del volumen (H3)**, no un patrón dirigido → no es una señal propia |
| H7 | **Externos más riesgosos** | ❌ descartada | Al revés: los 80 `External` son **más limpios** que los `Internal` (0% sin permiso y 0% fuera de horario, vs 0.98% y 1.24%). El `user_type` por sí solo no separa señal de ruido |

**Nota de calidad de datos:** la estructura es íntegra (sin huérfanos/duplicados/fechas imposibles). `resource_type`==`source_system` siempre → columna redundante. El riesgo está en el comportamiento, no en los datos.

> **Lectura de H6 y H7:** documentar las hipótesis que se *descartan* es parte del análisis. H6 parecía una señal pero resultó un subproducto del volumen, y H7 apuntaba en la dirección equivocada. Esto evita introducir ruido en cualquier modelo posterior.